<a href="https://colab.research.google.com/github/lgumieniak/Othello/blob/main/Othello.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install colorama

In [2]:
from colorama import Fore, Style
import sys
import math
import time

# Mise en place du jeu

def init_grille():
    # Initialise la grille de jeu vide avec les pions de départ
    grille = [[' ' for _ in range(8)] for _ in range(8)] # Grille format 8/8
    grille[3][3], grille[4][4], grille[3][4], grille[4][3] = '○', '○', '●', '●' # Position de départ des pions
    return grille

def mvts_valides(grille, joueur):
    # Trouve tous les mouvements valides pour le joueur actuel
    opposant = '○' if joueur == '●' else '●' # Définit le modèle de pion du joueur de l'IA
    deplacements = [(1, 0), (0, 1), (-1, 0), (0, -1), (1, 1), (-1, -1), (1, -1), (-1, 1)] # Définit les positions de déplacement (haut/bas/droite/gauche/diagonales..)
    valides = [] # liste contenant les cases valides

    for x in range(8):
        for y in range(8):
            if grille[x][y] != ' ': # parcours les cases de la grille en vérifiant si elles sont vides ou non
                continue
            for dx, dy in deplacements: # Si la case est pleine, recherche des pions adversaires grace aux directions définies dans "deplacements"
                nx, ny = x + dx, y + dy
                if 0 <= nx < 8 and 0 <= ny < 8 and grille[nx][ny] == opposant: # Si pièce adverse trouvée :
                    while 0 <= nx + dx < 8 and 0 <= ny + dy < 8 and grille[nx][ny] == opposant: # parcours la direction jusqu'à ce qu'une pièce du joueur actuel soit trouvée ou que les limites du plateau soient atteintes
                        nx += dx
                        ny += dy
                        if grille[nx][ny] == joueur:# Si une piece du joueur actuel est trouvée, ajout de la case dans liste "valides".
                            valides.append((x, y))
                            break
                    else:
                        continue
                    break
    return valides

def afficher_grille(grille, valides=None):
    if valides is None:
        valides = []

    print('     A   B   C   D   E   F   G   H')
    print(" ")
    for i, ligne in enumerate(grille):
        print(int(i+1), end=" | ")
        for j, case in enumerate(ligne):
            if (i, j) in valides:
                print(Fore.RED + ' +' + Style.RESET_ALL, end=' |')
            else:
                print(' ' + case, end=' |')
        print("\n  +--------------------------------+")

def jouer_coup(grille, joueur, coup):
    x, y = coup
    grille[x][y] = joueur
    opp = '○' if joueur == '●' else '●'
    dirs = [(1, 0), (0, 1), (-1, 0), (0, -1), (1, 1), (-1, -1), (1, -1), (-1, 1)]

    for dx, dy in dirs:
        nx, ny = x + dx, y + dy
        pions_a_retourner = []
        while 0 <= nx < 8 and 0 <= ny < 8 and grille[nx][ny] == opp:
            pions_a_retourner.append((nx, ny))
            nx += dx
            ny += dy
        if 0 <= nx < 8 and 0 <= ny < 8 and grille[nx][ny] == joueur:
            for px, py in pions_a_retourner:
                grille[px][py] = joueur


def j_joue(grille, joueur):
    # Demande au joueur humain de choisir un coup valide
    valides = mvts_valides(grille, joueur)
    if not valides:
        return None

    coup = None
    while coup not in valides:
        coup_str = input(f"Joueur {joueur}, entrez votre coup (ex: 3C): ").upper()
        if len(coup_str) != 2 or not coup_str[0].isdigit() or not coup_str[1].isalpha():
            print("Coup invalide. Veuillez réessayer.")
            continue

        y, x = ord(coup_str[1]) - ord('A'), int(coup_str[0]) - 1
        if x < 0 or x >= 8 or y < 0 or y >= 8:
            print("Coup invalide. Veuillez réessayer.")
            continue

        coup = (x, y)

        if coup not in valides:
            print("Coup invalide. Veuillez choisir parmi les coups valides.")
            continue

    return coup


def compter_pions(grille): # Compte simplement le score des deux joueurs
    pions_x = 0
    pions_o = 0

    for ligne in grille:
        for case in ligne:
            if case == '●':
                pions_x += 1
            elif case == '○':
                pions_o += 1

    return pions_x, pions_o

# Fonction d'heuristique

def heuristique(grille, joueur, profondeur):
    opp = '○' if joueur == '●' else '●'

    #Pose une valeur sur chaque case de la grille en fonction des stratégies qu'elles peuvent permettre
    #Exemple : le poids des coins est le plus grand car ce sont des pions qui ne peuvent plus changer derrière
    poids_position = [
        [120, -20, 20,  5,  5, 20, -20, 120],
        [-20, -40, -5, -5, -5, -5, -40, -20],
        [20,  -5, 15,  3,  3, 15,  -5,  20],
        [5,   -5,  3,  3,  3,  3,  -5,   5],
        [5,   -5,  3,  3,  3,  3,  -5,   5],
        [20,  -5, 15,  3,  3, 15,  -5,  20],
        [-20, -40, -5, -5, -5, -5, -40, -20],
        [120, -20, 20,  5,  5, 20, -20, 120],
    ]

    score_position = sum(grille[i][j] == joueur for i in range(8) for j in range(8)) - \
                     sum(grille[i][j] == opp for i in range(8) for j in range(8))
    score_position *= 2
    #score_position: un score qui reflète la position des pièces sur le plateau en utilisant une grille de poids appelée poids_position.

    score_mobility = len(mvts_valides(grille, joueur)) - len(mvts_valides(grille, opp))
    score_mobility *= 2
    #score_mobility: le nombre de mouvements valides pour le joueur actuel moins le nombre de mouvements valides pour l'adversaire, où le nombre de mouvements valides est trouvé à l'aide de la fonction mvts_valides.

    stable_pions = set([(0,0), (0,7), (7,0), (7,7)]).union(set([(0,1), (1,0), (1,1), (6,0), (6,1), (7,6), (6,6), (6,7)]))
    for i in range(1,7):
        stable_pions.add((0,i))
        stable_pions.add((i,0))
        stable_pions.add((i,7))
        stable_pions.add((7,i))
    for i in range(2,6):
        for j in range(2,6):
            if all(grille[i+k][j+l] == joueur for k,l in [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]):
                stable_pions.add((i,j))
            elif all(grille[i+k][j+l] == opp for k,l in [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]):
                stable_pions.add((i,j))

    # Calcul du nombre de pions stables pour chaque joueur
    stable_joueur = sum([grille[i][j] == joueur and (i, j) in stable_pions for i in range(8) for j in range(8)])
    stable_opp = sum([grille[i][j] == opp and (i, j) in stable_pions for i in range(8) for j in range(8)])

    # Score basé sur la stabilité des pions
    score_stabilite = (stable_joueur - stable_opp) * 100

    total_pions = sum(grille[i][j] != ' ' for i in range(8) for j in range(8)) # nombre total de pièces sur le plateau. En fonction du nombre total de pièces, la fonction retourne un score combinant les différents scores.

    # Chaque partie du jeu a une importance et un poids différents comme mentionné dans l'énoncé. En fonction de l'étape,
    # la fonction heuristique retournera une valeur différente en utilisant la somme des trois scores précédents, de deux ou d'un seul seulement.

    if total_pions < 16: # Première partie du jeu "Ouverture"
        return score_position + score_stabilite
    elif total_pions < 48: # Deuxième partie du jeu "Milieu de partie"
        return score_position + score_stabilite + 50 * score_mobility
    else: # Troisième partie du jeu "Fin de partie"
        return 100 * (sum(grille[i][j] == joueur for i in range(8) for j in range(8)) -
                      sum(grille[i][j] == opp for i in range(8) for j in range(8))) + score_mobility + score_stabilite

def minmax_alpha_beta(grille, joueur, profondeur, alpha, beta):
    if profondeur == 0:
        return heuristique(grille, joueur, profondeur), None

    opp = '○' if joueur == '●' else '●'
    coups_valides = mvts_valides(grille, joueur)

    if not coups_valides:
        return heuristique(grille, joueur, profondeur), None

    meilleur_coup = coups_valides[0]
    meilleur_score = -math.inf

    for coup in coups_valides:
        grille_copie = [ligne.copy() for ligne in grille]
        jouer_coup(grille_copie, joueur, coup)
        score, _ = minmax_alpha_beta(grille_copie, opp, profondeur - 1, -beta, -max(alpha, meilleur_score))
        score = -score
        if score > meilleur_score:
            meilleur_score = score
            meilleur_coup = coup

            if meilleur_score >= beta:
                break

    return meilleur_score, meilleur_coup

def coord_to_str_ia(coord):
    y, x = coord
    return f"{chr(x + ord('A'))}{y+1}"

def determiner_profondeur(grille):
   # Déterminer le nombre de pions actuels sur la grille
    total_pions = sum(grille[i][j] != ' ' for i in range(8) for j in range(8))

    # Ajuster la profondeur en fonction du stade de la partie
    if total_pions < 16:
        profondeur = 2         # Profondeur réduite pour le début de la partie
    elif total_pions < 48:
        profondeur = 4         # Profondeur moyenne pour le milieu de la partie
    else:
        profondeur = 8         # Profondeur augmentée pour la fin de la partie

    return profondeur

def ia_joue(grille, joueur):
    profondeur = determiner_profondeur(grille)
    score, meilleur_coup = minmax_alpha_beta(grille, joueur, profondeur, -math.inf, math.inf)
    i, j = coord_to_str_ia(meilleur_coup)
    print(f"L'IA joue en {j}{i}")
    return meilleur_coup

def commencer_partie():
    grille = init_grille()
    choix = ""
    while choix.lower() not in ["ia", "joueur"]:
        choix = input("Qui commence ? (IA/Joueur): ")
    joueur = "●"
    ia = choix.lower() == 'ia'
    return grille, joueur, ia

def main():
    grille, joueur, ia = commencer_partie()
    print(" ")
    print("Grille de départ :")
    print(" ")
    afficher_grille(grille)
    pions_x, pions_o = compter_pions(grille)
    print(f"Score : ● - {pions_x}, ○ - {pions_o}")
    print(" ")

    tps_ia = 0
    tps_j = 0

    while True:

        valides = mvts_valides(grille, joueur)
        if not valides :
            print(f"Aucun coup valide pour le joueur {joueur}, passez votre tour")
            print(" ")
            # vérifie si le joueur suivant a également des coups valides
            valides_suivant = mvts_valides(grille, '○' if joueur == '●' else '●')
            if not valides_suivant:
                print(f"Aucun coup valide pour le joueur {'○' if joueur == '●' else '●'}")
                print(" ")
                break
            if joueur == "●":
                joueur = "○"
            else:
                joueur = "●"
                if not ia:
                    continue

        # Modification pour afficher les coups valides en rouge
        afficher_grille(grille, valides)

        if ia:
            debut = time.time()
            coup = ia_joue(grille, joueur)
            fin = time.time()
            tps_ia += fin - debut
            print(f"Temps de calcul de l'IA : {tps_ia:.2f} secondes")
        else :
            debut = time.time()
            coup = j_joue(grille, joueur)
            fin = time.time()
            tps_j += fin - debut
            print(f"Temps du joueur : {tps_j:.2f} secondes")

        print(" ")

        if coup is None:
            if not mvts_valides(grille, '○' if joueur == '●' else '●'):
                break
        else:
            jouer_coup(grille, joueur, coup)
            afficher_grille(grille)
            pions_x, pions_o = compter_pions(grille)
            print(f"Score : ● - {pions_x}, ○ - {pions_o}")
            print(" ")

        joueur = '○' if joueur == '●' else '●'
        ia = not ia

        if pions_x + pions_o == 64:
            break

    print("Fin de la partie.")
    print(" ")
    afficher_grille(grille)

    pions_x, pions_o = compter_pions(grille)
    print(f"Score final : ● - {pions_x}, ○ - {pions_o}")
    print(f"Temps total de calcul de l'IA : {tps_ia:.2f} secondes")
    print(f"Temps total du joueur : {tps_j:.2f} secondes")



In [3]:
main()

Qui commence ? (IA/Joueur): IA
 
Grille de départ :
 
     A   B   C   D   E   F   G   H
 
1 |    |   |   |   |   |   |   |   |
  +--------------------------------+
2 |    |   |   |   |   |   |   |   |
  +--------------------------------+
3 |    |   |   |   |   |   |   |   |
  +--------------------------------+
4 |    |   |   | ○ | ● |   |   |   |
  +--------------------------------+
5 |    |   |   | ● | ○ |   |   |   |
  +--------------------------------+
6 |    |   |   |   |   |   |   |   |
  +--------------------------------+
7 |    |   |   |   |   |   |   |   |
  +--------------------------------+
8 |    |   |   |   |   |   |   |   |
  +--------------------------------+
Score : ● - 2, ○ - 2
 
     A   B   C   D   E   F   G   H
 
1 |    |   |   |   |   |   |   |   |
  +--------------------------------+
2 |    |   |   |   |   |   |   |   |
  +--------------------------------+
3 |    |   |   | + |   |   |   |   |
  +--------------------------------+
4 |    |   | + | ○ | ● |   |   |   